In [ ]:
import re
from pathlib import Path
from string import Template

from IPython.display import display

from pair_recon.app.config import env_value, load_environment
from pair_recon.app.runner import StatementSql, workspace
from pair_recon.core.metadata.workbook import (
    read_pair_workbook,
    upsert_pair_configuration,
)

# Load .pair_recon.env without replacing values already present in the process.
load_environment()
ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "setup/schema.sql").is_file()
)
PROFILE = env_value("DBX_PROFILE", default="corp")
WAREHOUSE_ID = env_value("DBX_WAREHOUSE_ID")
PAIR_TABLE = env_value("DBX_PAIR_TABLE", default="catalog.schema.pair_recon_pair_meta")
CATALOG, SCHEMA, _ = PAIR_TABLE.split(".")
WORKBOOK = Path(env_value("DBX_PAIR_WORKBOOK", default=str(ROOT / "setup/pairs.xlsx"))).expanduser()

if not WAREHOUSE_ID:
    raise ValueError("Set DBX_WAREHOUSE_ID in .pair_recon.env")

client = workspace(PROFILE)
sql = StatementSql(client, WAREHOUSE_ID)

In [ ]:
# Keep identifiers unquoted and conservative because they are substituted into SQL text.
IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_$]*$")
SQL_TYPE = re.compile(r"^[A-Za-z][A-Za-z0-9_]*(?:<[A-Za-z0-9_, <>]+>)?$")
TABLES = (
    "pair_recon_pair_meta",
    "pair_recon_column_map",
    "pair_recon_pair_sql",
    "pair_recon_run_status",
    "pair_recon_run_error",
    "pair_recon_progress_snapshot",
)


def identifier(value):
    text = str(value).strip()
    if not IDENTIFIER.fullmatch(text):
        raise ValueError(f"Invalid Unity Catalog identifier: {value!r}")
    return text


def table_name(table):
    name = identifier(table)
    if name not in TABLES:
        raise ValueError(f"Unknown PairRecon table: {name}")
    return Template("$catalog.$schema.$table").substitute(
        catalog=identifier(CATALOG),
        schema=identifier(SCHEMA),
        table=name,
    )


def schema_statements():
    rendered = Template((ROOT / "setup/schema.sql").read_text(encoding="utf-8")).substitute(
        catalog=identifier(CATALOG),
        schema=identifier(SCHEMA),
    )
    return [statement.strip() for statement in rendered.split(";") if statement.strip()]


display({"profile": PROFILE, "warehouse_id": WAREHOUSE_ID, "catalog": CATALOG, "schema": SCHEMA})

In [ ]:
def create_tables():
    # CREATE TABLE IF NOT EXISTS keeps existing data and table definitions intact.
    statements = schema_statements()
    for statement in statements:
        sql.execute(statement)
    return {"created_or_existing": len(statements), "tables": list(TABLES)}


def drop_tables():
    # This is destructive; tables are dropped in reverse dependency order.
    dropped = []
    for table in reversed(TABLES):
        statement = Template("DROP TABLE IF EXISTS $table").substitute(
            table=table_name(table)
        )
        sql.execute(statement)
        dropped.append(table)
    return {"dropped": dropped}


def delete_rows(table, key_values):
    # Values use StatementSql parameters; only validated identifiers enter the SQL text.
    if not key_values:
        raise ValueError("key_values cannot be empty")
    predicates = []
    parameters = {}
    for index, (column, value) in enumerate(key_values.items()):
        column = identifier(column)
        parameter = f"key_{index}"
        predicates.append(f"{column} = :{parameter}")
        parameters[parameter] = value
    statement = Template("DELETE FROM $table WHERE $predicate").substitute(
        table=table_name(table),
        predicate=" AND ".join(predicates),
    )
    sql.execute(statement, parameters)
    return {"table": table, "deleted_by": dict(key_values)}

In [ ]:
def upsert_configuration():
    # The project helper validates pairs.xlsx and MERGEs pair_meta plus pair_sql.
    rows = read_pair_workbook(WORKBOOK)
    return upsert_pair_configuration(
        sql,
        table_name("pair_recon_pair_meta"),
        rows,
    )


def merge_rows(table, key_columns, rows, casts=None):
    # Use this generic helper for simple row MERGEs; pair metadata should use upsert_configuration().
    if not rows:
        return {"table": table, "rows": 0}
    columns = tuple(rows[0])
    if not columns or any(tuple(row) != columns for row in rows):
        raise ValueError("Every MERGE row must have the same ordered columns")
    columns = tuple(identifier(column) for column in columns)
    keys = tuple(identifier(column) for column in key_columns)
    if not keys or not set(keys).issubset(columns):
        raise ValueError("key_columns must be a non-empty subset of row columns")
    updates = tuple(column for column in columns if column not in keys)
    if not updates:
        raise ValueError("MERGE requires at least one non-key column")
    cast_types = {identifier(column): str(value).strip() for column, value in (casts or {}).items()}
    if not set(cast_types).issubset(columns) or any(
        not SQL_TYPE.fullmatch(value) for value in cast_types.values()
    ):
        raise ValueError("casts contains an invalid column or SQL type")
    for row_index, row in enumerate(rows):
        parameters = {f"r{row_index}_{column}": row[column] for column in columns}
        source_fields = []
        for column in columns:
            expression = f":r{row_index}_{column}"
            if column in cast_types:
                expression = f"CAST({expression} AS {cast_types[column]})"
            source_fields.append(f"{expression} AS {column}")
        statement = Template(
            """MERGE INTO $table AS t
USING (SELECT $source_fields) AS s ON $match
WHEN MATCHED THEN UPDATE SET $updates
WHEN NOT MATCHED THEN INSERT ($columns) VALUES ($values)"""
        ).substitute(
            table=table_name(table),
            source_fields=", ".join(source_fields),
            match=" AND ".join(f"t.{column} = s.{column}" for column in keys),
            updates=", ".join(f"t.{column} = s.{column}" for column in updates),
            columns=", ".join(columns),
            values=", ".join(f"s.{column}" for column in columns),
        )
        sql.execute(statement, parameters)
    return {"table": table, "rows": len(rows)}

In [ ]:
# Keep all destructive or remote writes disabled until the values above have been reviewed.
CREATE_TABLES = False
UPSERT_CONFIGURATION = False
DROP_TABLES = False

if sum((CREATE_TABLES, UPSERT_CONFIGURATION, DROP_TABLES)) > 1:
    raise ValueError("Enable only one bulk action at a time")
if CREATE_TABLES:
    display(create_tables())
elif UPSERT_CONFIGURATION:
    display(upsert_configuration())
elif DROP_TABLES:
    display(drop_tables())
else:
    display({"preview_create": schema_statements(), "workbook": str(WORKBOOK)})

# Example targeted delete:
# delete_rows("pair_recon_column_map", {"pair_name": "orders", "left_column": "AMOUNT"})

# Example generic MERGE:
# merge_rows(
#     "pair_recon_column_map",
#     ("pair_name", "left_column"),
#     [{"pair_name": "orders", "left_column": "AMOUNT", "right_column": "amount", "data_type": "numeric"}],
# )